In [1]:
import polars as pl

from ohlc_dss_model.data import load_parquet

In [2]:
df = load_parquet()
print(df.head(5))

shape: (5, 6)
┌─────────────────────────┬─────────┬─────────┬─────────┬─────────┬────────┐
│ DateTime                ┆ Open    ┆ High    ┆ Low     ┆ Close   ┆ Volume │
│ ---                     ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---    │
│ datetime[ns, UTC]       ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ u64    │
╞═════════════════════════╪═════════╪═════════╪═════════╪═════════╪════════╡
│ 2016-01-03 23:00:00 UTC ┆ 4592.5  ┆ 4606.75 ┆ 4592.5  ┆ 4603.25 ┆ 1134   │
│ 2016-01-03 23:30:00 UTC ┆ 4603.0  ┆ 4603.0  ┆ 4597.5  ┆ 4600.5  ┆ 425    │
│ 2016-01-04 00:00:00 UTC ┆ 4600.25 ┆ 4604.0  ┆ 4599.75 ┆ 4604.0  ┆ 477    │
│ 2016-01-04 00:30:00 UTC ┆ 4603.5  ┆ 4604.25 ┆ 4602.25 ┆ 4603.75 ┆ 331    │
│ 2016-01-04 01:00:00 UTC ┆ 4603.75 ┆ 4605.5  ┆ 4598.75 ┆ 4599.0  ┆ 555    │
└─────────────────────────┴─────────┴─────────┴─────────┴─────────┴────────┘


First, we need to handle time zone alignment and account for Daylight Saving Time (DST) changes. Since we've known that the timeone is in UTC conversion should be straight forward

In [5]:
df = df.with_columns(
    pl.col("DateTime")
    .dt.replace_time_zone("UTC")
    .dt.convert_time_zone("America/New_York")
    .alias("DateTime_NY")
)

print(df.select(["DateTime", "DateTime_NY"]).head(5))

shape: (5, 2)
┌─────────────────────────┬────────────────────────────────┐
│ DateTime                ┆ DateTime_NY                    │
│ ---                     ┆ ---                            │
│ datetime[ns, UTC]       ┆ datetime[ns, America/New_York] │
╞═════════════════════════╪════════════════════════════════╡
│ 2016-01-03 23:00:00 UTC ┆ 2016-01-03 18:00:00 EST        │
│ 2016-01-03 23:30:00 UTC ┆ 2016-01-03 18:30:00 EST        │
│ 2016-01-04 00:00:00 UTC ┆ 2016-01-03 19:00:00 EST        │
│ 2016-01-04 00:30:00 UTC ┆ 2016-01-03 19:30:00 EST        │
│ 2016-01-04 01:00:00 UTC ┆ 2016-01-03 20:00:00 EST        │
└─────────────────────────┴────────────────────────────────┘


In [6]:
dst_test = df.filter(
    (pl.col("DateTime_NY").dt.date() == pl.date(2023, 3, 10))
    | (pl.col("DateTime_NY").dt.date() == pl.date(2023, 3, 15))
).filter(pl.col("DateTime_NY").dt.time() == pl.time(9, 30))

print(dst_test.select(["DateTime", "DateTime_NY"]))

shape: (2, 2)
┌─────────────────────────┬────────────────────────────────┐
│ DateTime                ┆ DateTime_NY                    │
│ ---                     ┆ ---                            │
│ datetime[ns, UTC]       ┆ datetime[ns, America/New_York] │
╞═════════════════════════╪════════════════════════════════╡
│ 2023-03-10 14:30:00 UTC ┆ 2023-03-10 09:30:00 EST        │
│ 2023-03-15 13:30:00 UTC ┆ 2023-03-15 09:30:00 EDT        │
└─────────────────────────┴────────────────────────────────┘


As demonstrated, DST shifts are now correctly handled, and the dataset's timestamps are ready for further analysis.